# ECSC Developmental Sentence Influence Analysis

**Question:** Does the density of long-range sentence-level influence increase with age in children's spontaneous speech?

**Method:** Leave-one-sentence-out influence matrices on ECSC frog story narrations. Same approach as RAID v6 but with age as the key variable.

**Data:** 341 frog story narrations from children aged 5-11. Spontaneous speech — no pre-planning.

**Predictions:**
- Similar local influence (distance 1-3) across ages
- More long-range hotspots in older children
- Higher hotspot density at distance 10+ as age increases

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/ecsc_processed")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/ECSC_influence")
    if (DRIVE_DATA / "transcripts.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/ecsc_processed")
        if not (LOCAL_DATA / "transcripts.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            print("Upload transcripts.jsonl:")
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/ecsc_influence")
    DATA_DIR = Path("../data/ecsc_processed")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

# Sample size and sentence cap — balance depth vs runtime
N_DOCS_PER_BIN = 30  # docs per age bin
MAX_SENTENCES = 20   # cap per doc (O(n^2) cost)
RANDOM_SEED = 42
MIN_WORDS = 150

AGE_BINS = [
    (59, 78, '5-6y'),
    (79, 96, '7-8y'),
    (97, 138, '9-11y'),
]

print(f"Docs per age bin: {N_DOCS_PER_BIN}")
print(f"Max sentences per doc: {MAX_SENTENCES}")
print(f"Age bins: {[b[2] for b in AGE_BINS]}")

In [ ]:
corpus_all = []
with open(DATA_DIR / "transcripts.jsonl") as f:
    for line in f:
        d = json.loads(line)
        pop = json.loads(d['population'])
        d['age_months'] = pop['age_months']
        d['sex'] = pop['sex']
        d['word_count'] = len(d['text'].split())
        corpus_all.append(d)

# Filter and assign age bins
corpus_all = [d for d in corpus_all if d['word_count'] >= MIN_WORDS]
for d in corpus_all:
    for lo, hi, label in AGE_BINS:
        if lo <= d['age_months'] <= hi:
            d['age_bin'] = label
            break

# Sample N per age bin
rng = np.random.RandomState(RANDOM_SEED)
corpus = []
for lo, hi, label in AGE_BINS:
    pool = [d for d in corpus_all if d.get('age_bin') == label]
    n = min(len(pool), N_DOCS_PER_BIN)
    chosen = rng.choice(pool, size=n, replace=False)
    corpus.extend(chosen)

print(f"Selected {len(corpus)} transcripts")
for lo, hi, label in AGE_BINS:
    sub = [d for d in corpus if d['age_bin'] == label]
    ages = [d['age_months'] for d in sub]
    print(f"  {label}: n={len(sub)}, age range={min(ages)}-{max(ages)} months")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip().split()) >= 4]


@torch.no_grad()
def compute_ppl_on_target(context_token_ids, target_token_ids):
    full_ids = context_token_ids + target_token_ids
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    target_start = len(context_token_ids)
    total_loss = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[full_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_influence_matrix(doc):
    sentences = split_into_sentences(doc['text'])
    if len(sentences) < 4:
        return None, None

    if len(sentences) > MAX_SENTENCES:
        sentences = sentences[:MAX_SENTENCES]

    n = len(sentences)
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sentences]
    matrix = np.full((n, n), np.nan)

    for t in range(2, n):
        target_ids = sent_ids[t]
        if len(target_ids) < 3:
            continue

        full_context_ids = []
        for k in range(t):
            full_context_ids.extend(sent_ids[k])

        ppl_full = compute_ppl_on_target(full_context_ids, target_ids)
        if math.isinf(ppl_full) or ppl_full <= 0:
            continue

        for i in range(t):
            dropped_context_ids = []
            for k in range(t):
                if k != i:
                    dropped_context_ids.extend(sent_ids[k])
            if len(dropped_context_ids) < 2:
                continue

            ppl_dropped = compute_ppl_on_target(dropped_context_ids, target_ids)
            if math.isinf(ppl_dropped) or ppl_dropped <= 0:
                continue

            matrix[i, t] = math.log(ppl_dropped / ppl_full)

    return sentences, matrix


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "ecsc_influence_matrices.npz"
meta_path = BASE_DIR / "ecsc_influence_meta.json"

if results_path.exists() and meta_path.exists():
    loaded = np.load(results_path, allow_pickle=True)
    all_matrices = list(loaded['matrices'])
    with open(meta_path) as f:
        all_meta = json.load(f)
    print(f"Loaded {len(all_matrices)} influence matrices")
else:
    all_matrices = []
    all_meta = []

    for doc in tqdm(corpus, desc="Computing influence matrices"):
        sentences, matrix = compute_influence_matrix(doc)
        if matrix is None:
            continue
        all_matrices.append(matrix)
        all_meta.append({
            'doc_id': doc['doc_id'],
            'age_months': doc['age_months'],
            'age_bin': doc['age_bin'],
            'sex': doc['sex'],
            'n_sentences': len(sentences),
            'sentences': sentences,
        })

    np.savez(results_path, matrices=np.array(all_matrices, dtype=object))
    with open(meta_path, 'w') as f:
        json.dump(all_meta, f)
    print(f"Computed {len(all_matrices)} matrices, saved to {BASE_DIR}")

print(f"Documents: {len(all_matrices)}")
for lo, hi, label in AGE_BINS:
    n = sum(1 for m in all_meta if m['age_bin'] == label)
    print(f"  {label}: {n}")

In [ ]:
# Aggregate influence by distance AND age bin
by_dist_age = {label: {} for _, _, label in AGE_BINS}

for doc_idx, matrix in enumerate(all_matrices):
    n = matrix.shape[0]
    age_bin = all_meta[doc_idx]['age_bin']
    for i in range(n):
        for t in range(i+1, n):
            val = matrix[i, t]
            if not np.isnan(val):
                d = t - i
                if d not in by_dist_age[age_bin]:
                    by_dist_age[age_bin][d] = []
                by_dist_age[age_bin][d].append(val)

print("INFLUENCE BY SENTENCE DISTANCE: BY AGE")
print(f"{'Dist':>5}", end='')
for _, _, label in AGE_BINS:
    print(f"  {label+' mean':>10} {label+' %+':>8} {label+' n':>6}", end='')
print()
print("-" * 85)
for d in range(1, 20):
    print(f"{d:>5}", end='')
    for _, _, label in AGE_BINS:
        vals = np.array(by_dist_age[label].get(d, []))
        if len(vals) >= 3:
            print(f"  {vals.mean():>10.4f} {100*np.mean(vals>0):>7.1f}% {len(vals):>6}", end='')
        else:
            print(f"  {'':>10} {'':>8} {'':>6}", end='')
    print()

print()
print("HOTSPOT DENSITY BY AGE:")
for threshold in [0.05, 0.1]:
    for min_dist in [5, 10]:
        print(f"  influence>{threshold}, dist>={min_dist}:")
        for _, _, label in AGE_BINS:
            count = total = 0
            for d in by_dist_age[label]:
                if d >= min_dist:
                    vals = np.array(by_dist_age[label][d])
                    count += np.sum(vals > threshold)
                    total += len(vals)
            pct = 100*count/total if total > 0 else 0
            print(f"    {label}: {count}/{total} = {pct:.1f}%")

print()
print("SENTENCE LIFESPAN BY AGE:")
for _, _, label in AGE_BINS:
    lifespans = []
    for doc_idx, matrix in enumerate(all_matrices):
        if all_meta[doc_idx]['age_bin'] != label:
            continue
        n = matrix.shape[0]
        for i in range(n - 2):
            last_d = 0
            for t in range(i+1, n):
                val = matrix[i, t]
                if not np.isnan(val) and val > 0.01:
                    last_d = t - i
            if last_d > 0:
                lifespans.append(last_d)
    lifespans = np.array(lifespans)
    print(f"  {label}: mean={lifespans.mean():.1f}, median={np.median(lifespans):.0f}, "
          f">5: {100*np.mean(lifespans>5):.1f}%, >10: {100*np.mean(lifespans>10):.1f}%")

print()
print("AGE CORRELATION WITH MEAN INFLUENCE AT EACH DISTANCE:")
for d in [1, 3, 5, 8, 10, 15]:
    # Per-doc mean influence at this distance
    doc_vals = []
    doc_ages = []
    for doc_idx, matrix in enumerate(all_matrices):
        n = matrix.shape[0]
        infs = []
        for i in range(n):
            t = i + d
            if t < n and not np.isnan(matrix[i, t]):
                infs.append(matrix[i, t])
        if len(infs) >= 2:
            doc_vals.append(np.mean(infs))
            doc_ages.append(all_meta[doc_idx]['age_months'])
    if len(doc_vals) >= 10:
        r, p = stats.pearsonr(doc_ages, doc_vals)
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
        print(f"  dist={d:>2}: r={r:+.3f}, p={p:.4f} {sig} (n={len(doc_vals)})")

In [ ]:
age_colors = {'5-6y': '#e74c3c', '7-8y': '#f39c12', '9-11y': '#27ae60'}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Decay curves by age
ax = axes[0, 0]
for _, _, label in AGE_BINS:
    dists = sorted([d for d in by_dist_age[label] if len(by_dist_age[label][d]) >= 5])
    means = [np.mean(by_dist_age[label][d]) for d in dists]
    sems = [np.std(by_dist_age[label][d])/np.sqrt(len(by_dist_age[label][d])) for d in dists]
    n_docs = sum(1 for m in all_meta if m['age_bin'] == label)
    ax.errorbar(dists, means, yerr=sems, fmt='o-', color=age_colors[label],
                linewidth=2, markersize=5, capsize=2, label=f'{label} (n={n_docs})')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Sentence Distance')
ax.set_ylabel('Mean Influence')
ax.set_title('A. Influence Decay by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Hotspot density by age and distance
ax = axes[0, 1]
threshold = 0.05
for _, _, label in AGE_BINS:
    dists = sorted([d for d in by_dist_age[label] if len(by_dist_age[label][d]) >= 5])
    pcts = []
    for d in dists:
        vals = np.array(by_dist_age[label][d])
        pcts.append(100 * np.mean(vals > threshold))
    ax.plot(dists, pcts, 'o-', color=age_colors[label], linewidth=2, markersize=5, label=label)
ax.set_xlabel('Sentence Distance')
ax.set_ylabel(f'% Pairs with Influence > {threshold}')
ax.set_title('B. Long-Range Hotspot Density by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# C: Age (continuous) vs mean long-range influence (dist >= 5)
ax = axes[1, 0]
doc_lr_influence = []
doc_ages = []
for doc_idx, matrix in enumerate(all_matrices):
    n = matrix.shape[0]
    lr_vals = []
    for i in range(n):
        for t in range(i+5, n):
            val = matrix[i, t]
            if not np.isnan(val):
                lr_vals.append(val)
    if len(lr_vals) >= 3:
        doc_lr_influence.append(np.mean(lr_vals))
        doc_ages.append(all_meta[doc_idx]['age_months'])

for _, _, label in AGE_BINS:
    mask = [all_meta[i]['age_bin'] == label for i in range(len(all_meta)) 
            if len([matrix[j,t] for j in range(all_matrices[i].shape[0]) 
                   for t in range(j+5, all_matrices[i].shape[0]) 
                   if not np.isnan(all_matrices[i][j,t])]) >= 3]

ax.scatter(doc_ages, doc_lr_influence, alpha=0.5, s=20, c=[age_colors[all_meta[i]['age_bin']] 
          for i in range(len(all_meta)) if i < len(doc_lr_influence)])
r, p = stats.pearsonr(doc_ages[:len(doc_lr_influence)], doc_lr_influence)
x_line = np.array([min(doc_ages), max(doc_ages)])
slope, intercept, _, _, _ = stats.linregress(doc_ages[:len(doc_lr_influence)], doc_lr_influence)
ax.plot(x_line, intercept + slope * x_line, 'k--', linewidth=2)
ax.set_xlabel('Age (months)')
ax.set_ylabel('Mean Influence (dist >= 5)')
ax.set_title(f'C. Long-Range Influence vs Age (r={r:.3f}, p={p:.4f})', fontweight='bold')
ax.grid(True, alpha=0.2)

# D: Example heatmaps — youngest vs oldest
ax = axes[1, 1]
# Find a good young and old example
young_idx = max([i for i, m in enumerate(all_meta) if m['age_bin'] == '5-6y'],
                key=lambda i: all_matrices[i].shape[0])
old_idx = max([i for i, m in enumerate(all_meta) if m['age_bin'] == '9-11y'],
              key=lambda i: all_matrices[i].shape[0])

# Show side by side using split axis
matrix_y = all_matrices[young_idx]
matrix_o = all_matrices[old_idx]
n_show = min(matrix_y.shape[0], matrix_o.shape[0], 15)
combined = np.zeros((n_show, n_show * 2 + 1))
combined[:, :n_show] = np.where(np.isnan(matrix_y[:n_show, :n_show]), 0, matrix_y[:n_show, :n_show])
combined[:, n_show] = np.nan  # separator
combined[:, n_show+1:] = np.where(np.isnan(matrix_o[:n_show, :n_show]), 0, matrix_o[:n_show, :n_show])

im = ax.imshow(combined, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
ax.axvline(n_show - 0.5, color='black', linewidth=2)
ax.set_xlabel('Target Sentence')
ax.set_ylabel('Source Sentence')
meta_y = all_meta[young_idx]
meta_o = all_meta[old_idx]
ax.set_title(f'D. Young ({meta_y["age_months"]}mo) vs Old ({meta_o["age_months"]}mo)', fontweight='bold')
plt.colorbar(im, ax=ax, label='Influence', shrink=0.8)

plt.suptitle('Developmental Sentence Influence: Does Long-Range Structure Increase with Age?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_developmental_influence.png', dpi=150, bbox_inches='tight')
plt.show()